# Sinistros de Trânsito e Infraestrutura Semafórica em Fortaleza
**Aluno:** Yuri Lucas Luz da Silva — Matrícula: 2223481

In [3]:
import warnings; warnings.filterwarnings('ignore')
import os, requests, json
import geopandas as gpd
import pandas as pd

os.makedirs('data', exist_ok=True)

URLS = {
    'data/sinistros.geojson': 'https://dados.fortaleza.ce.gov.br/dataset/76c32db6-97b4-439a-9f12-c0f6413b2013/resource/75f7e594-cab6-4907-b682-9b6af11ebc2c/download/sinistros-2015-2024.geojson',
    'data/semaforos.geojson': 'https://dados.fortaleza.ce.gov.br/dataset/c529ba45-27ae-4d3a-8f70-76019a87edba/resource/4e4a18a1-86cc-48a1-b631-3ef636b455ee/download/dadosabertos_semaforosctafor.geojson',
}

for path, url in URLS.items():
    if not os.path.exists(path):
        print(f'Baixando {os.path.basename(path)}...')
        r = requests.get(url, verify=False, stream=True, timeout=120)
        with open(path, 'wb') as f:
            for chunk in r.iter_content(256*1024): f.write(chunk)
        print(f'  {os.path.getsize(path)/1e6:.1f} MB')
    else:
        print(f'OK  {os.path.basename(path)}')

OK  sinistros.geojson
OK  semaforos.geojson


---
## 1. Sinistros de Trânsito (2015–2024)

In [ ]:
sinistros = gpd.read_file('data/sinistros.geojson')

# Aparentemente, a coluna HORA não é exportada pelo 
# geopandas (Skipping field HORA: unsupported OGR type: 10), 
# então vamos extrair diretamente do GeoJSON e adicionar ao GeoDataFrame
with open('data/sinistros.geojson') as f:
    feats = json.load(f)['features']
sinistros['HORA'] = [ft['properties'].get('HORA') for ft in feats]

print(f'Registros : {len(sinistros):,}')
print(f'Colunas   : {len(sinistros.columns)}')
print(f'Período   : {sinistros.ANO.min()} – {sinistros.ANO.max()}')

Skipping field HORA: unsupported OGR type: 10


Registros : 130,823
Colunas   : 28
Período   : 2015 – 2024


In [25]:
# campos e exemplos de valores.
# Removi a coluna geometry para não poluir a visualização, mas ela está presente no GeoDataFrame.
sample = sinistros.drop(columns='geometry').sample(1).squeeze()
sample_table = pd.DataFrame({
    'valor': sample,
    '%_ausentes': sinistros.drop(columns='geometry').isnull().mean().mul(100).round(1),
    'únicos': sinistros.drop(columns='geometry').nunique(),
    'tipo': sinistros.drop(columns='geometry').dtypes,
})
sample_table.reset_index().rename(columns={'index': 'coluna'})

,coluna,valor,%_ausentes,únicos,tipo
0,COD_ACIDENTE,363428,0.0,130820,object
1,DIA,14,0.0,31,int32
2,MES,10,0.0,12,int32
3,ANO,2020,0.0,10,int32
4,LOG1,"RUA BAR RIO BRANCO, 1998",0.0,41243,object
5,LOG2,",",0.0,7793,object
6,NUMERO,1998.0,50.0,6225,float64
7,LATITUDE,-3.735971,0.0,72066,float64
8,LONGITUDE,-38.531298,0.0,71201,float64
9,TIPOCRUZAMENTO,Meio de Quadra,0.0,14,object


In [6]:
# valores únicos das colunas categóricas relevantes
for col in ['SEVERIDADE', 'NATUREZA', 'TIPOCRUZAMENTO', 'CONTROLETRAFEGO', 'USOSOLO']:
    print(f'{col}: {sorted(sinistros[col].dropna().unique().tolist())}')

SEVERIDADE: ['Fatal', 'Ferido', 'Ileso', 'Não informado']
NATUREZA: ['ACIDENTE ONIBUS (TRANSPORTE COLETIVO)', 'Acidente Pessoal', 'Acidente com transporte ferroviário', 'Atropelamento', 'Atropelamento de Animal', 'Capotamento', 'Choque', 'Choque c/ Obstáculo Fixo', 'Colisão', 'Colisão Frontal', 'Colisão Lateral', 'Colisão Lateral Sentido Oposto', 'Colisão Transversal', 'Colisão Traseira', 'Engavetamento', 'Não informado', 'Outros', 'Queda', 'Queda de moto', 'Tombamento']
TIPOCRUZAMENTO: ['Com Via Férrea', 'Com via férrea', 'Cruz', 'Cruzamento', 'Duplo T', 'Meio De Quadra', 'Meio de Quadra', 'Meio de quadra', 'Não Informado', 'Não informado', 'Outros', 'Rotatória', 'T', 'Y']
CONTROLETRAFEGO: ['Cancela', 'Cancela Danificada', 'Dê Preferência', 'Faixa De Pedestre', 'Faixa de pedestre', 'Gesto', 'Não Há Controle', 'Não Informado', 'Não há Controle', 'Não informado', 'Outros', 'Pare', 'Semáforo', 'Semáforo Danificado', 'Semáforo Intermitente', 'Semáforo Manual']
USOSOLO: ['Cruzamento', 'Lon

---
## 1.1. Parque Semafórico

In [7]:
semaforos = gpd.read_file('data/semaforos.geojson')

print(f'Registros : {len(semaforos):,}')
print(f'Colunas   : {len(semaforos.columns)}')

Registros : 1,231
Colunas   : 26


In [26]:
# campos e exemplos de valores.
# Removi a coluna geometry para não poluir a visualização, mas ela está presente no GeoDataFrame.
sample_s = semaforos.drop(columns='geometry').sample(1).squeeze()
sample_table_s = pd.DataFrame({
    'valor': sample_s,
    '%_ausentes': semaforos.drop(columns='geometry').isnull().mean().mul(100).round(1),
    'únicos': semaforos.drop(columns='geometry').nunique(),
    'tipo': semaforos.drop(columns='geometry').dtypes,
})
sample_table_s.reset_index().rename(columns={'index': 'coluna'})

,coluna,valor,%_ausentes,únicos,tipo
0,CÓDIGO,950,0.0,1225,object
1,CRUZAMENTO,R. FRANCISCO CALAÇA & AV. FRANCISCO SÁ,0.0,1217,object
2,STATUS,CONVENCIONAL,0.0,5,string[python]
3,DATA_IMPLANTAÇÃO,2017-12-07 00:00:00,30.5,707,datetime64[ns]
4,DATA_DESATIVAÇÃO,NaT,88.3,120,datetime64[ns]
5,DATA_REATIVAÇÃO,NaT,95.4,45,datetime64[ns]
6,GRUPO,GR 98,11.5,180,object
7,MODO_CONTROLE,LOCAL,0.0,7,string[python]
8,QUANT_GF_T,3.0,11.6,10,float64
9,QUANT_GF_I,2.0,11.6,7,float64


In [27]:
# valores únicos das colunas de configuração
col_est = next((c for c in semaforos.columns if 'EST' in c.upper() and 'GIO' in c.upper()), None)
for col in ['STATUS', 'MODO_CONTROLE'] + ([col_est] if col_est else []):
    print(f'{col}: {sorted(semaforos[col].dropna().unique().tolist())}')

STATUS: ['CENTRALIZADO', 'CONVENCIONAL', 'DESATIVADO', 'DETRAN', 'PROJETO']
MODO_CONTROLE: ['Desconhecido', 'ECOTRAFIX', 'ECOTRAFIX CONJUGADO', 'LOCAL', 'LOCAL CONJUGADO', 'SCOOT', 'SCOOT CONJUGADO']
ESTÁGIOS: [2.0, 3.0, 4.0]


---
## 2. Integração com o Parque Semafórico e Limpeza dos Dados

A partir daqui (Parte 2) enriqueci os sinistros com o parque semafórico (fonte externa) e
preparei um dataset limpo. Cada bloco faz uma etapa: limpeza dos sinistros, limpeza dos semáforos,
reprojeção métrica, reconstrução temporal do parque, integração espacial e o dataset final.

In [ ]:
import numpy as np

# Parâmetros da integração

# Os dois datasets possuem o mesmo padrão (SIRGAS 2000) que está descrito no próprio dataframe.
# Para medir distâncias em metros, precisamos reprojetar para um CRS métrico adequado para Fortaleza,
# ou seja, usando o UTM 24S que é o timezone local. O EPSG correspondente é 31984. 
CRS_METRICO = 31984           # SIRGAS 2000 / UTM 24S (metros) — adequado para Fortaleza
RAIO_INTERSECCAO_M = 30       # distância máx. para considerar "semáforo na interseção"
RAIOS_DENSIDADE = (100, 250)  # raios (m) para contar semáforos ao redor de cada sinistro

In [28]:
# Confirmação do CRS de origem, lido do próprio arquivo: EPSG:4674 = SIRGAS 2000 (em graus).
# (ainda NÃO reprojetado aqui — a reprojeção para 31984 acontece no bloco 2.3)
print('Sinistros ->', sinistros.crs.name, '| EPSG:', sinistros.crs.to_epsg())
print('Semáforos ->', semaforos.crs.name, '| EPSG:', semaforos.crs.to_epsg())

Sinistros -> SIRGAS 2000 | EPSG: 4674
Semáforos -> SIRGAS 2000 / UTM zone 24S | EPSG: 31984


### 2.1 Limpeza dos sinistros

Padronizei as categóricas com variações de caixa/acento, montei a `data_hora` e removi a
coluna `NUMERO` (~50% ausente). Todas as linhas são mantidas.

In [11]:
# Normaliza espaços e padroniza variações de caixa/acento (variante -> rótulo único)
def normaliza_texto(serie):
    return serie.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)

MAPA_TIPOCRUZAMENTO = {
    'Com via férrea': 'Com Via Férrea',
    'Meio De Quadra': 'Meio de Quadra', 'Meio de quadra': 'Meio de Quadra',
    'Não Informado': 'Não informado',
}
MAPA_CONTROLETRAFEGO = {
    'Faixa De Pedestre': 'Faixa de Pedestre', 'Faixa de pedestre': 'Faixa de Pedestre',
    'Não há Controle': 'Não Há Controle', 'Não Informado': 'Não informado',
}
MAPA_NATUREZA = {'ACIDENTE ONIBUS (TRANSPORTE COLETIVO)': 'Acidente Ônibus (Transporte Coletivo)'}

sinistros['TIPOCRUZAMENTO']  = normaliza_texto(sinistros['TIPOCRUZAMENTO']).replace(MAPA_TIPOCRUZAMENTO)
sinistros['CONTROLETRAFEGO'] = normaliza_texto(sinistros['CONTROLETRAFEGO']).replace(MAPA_CONTROLETRAFEGO)
sinistros['NATUREZA']        = normaliza_texto(sinistros['NATUREZA']).replace(MAPA_NATUREZA)
sinistros['USOSOLO']         = normaliza_texto(sinistros['USOSOLO'])
sinistros['SEVERIDADE']      = normaliza_texto(sinistros['SEVERIDADE'])

# data_hora a partir de DIA/MES/ANO + HORA (datas impossíveis viram NaT)
base = pd.to_datetime(dict(year=sinistros['ANO'], month=sinistros['MES'], day=sinistros['DIA']), errors='coerce')
hora = pd.to_timedelta(sinistros['HORA'].fillna('00:00:00'), errors='coerce').fillna(pd.Timedelta(0))
sinistros['data_hora'] = base + hora

# NUMERO: ~50% ausente e pouco útil -> remove
sinistros = sinistros.drop(columns=['NUMERO'])

print('Limpeza dos sinistros concluída |', f'{len(sinistros):,}', 'linhas')
print('Categorias em TIPOCRUZAMENTO após padronização:', sinistros['TIPOCRUZAMENTO'].nunique())

Limpeza dos sinistros concluída | 130,823 linhas
Categorias em TIPOCRUZAMENTO após padronização: 10


### 2.2 Limpeza dos semáforos

Converti as datas (formato DD/MM/AAAA), tratei `ESTÁGIOS` (mantendo ausentes) e padronizei
`STATUS`/`MODO_CONTROLE` (ausente vira "Desconhecido").

In [12]:
for col in ['DATA_IMPLANTAÇÃO', 'DATA_DESATIVAÇÃO', 'DATA_REATIVAÇÃO']:
    semaforos[col] = pd.to_datetime(semaforos[col], dayfirst=True, errors='coerce')

semaforos['estagios_num'] = semaforos['ESTÁGIOS'].astype('Int64')
semaforos['STATUS'] = normaliza_texto(semaforos['STATUS'])
semaforos['MODO_CONTROLE'] = normaliza_texto(semaforos['MODO_CONTROLE']).fillna('Desconhecido')

print('Limpeza dos semáforos concluída')
print('ESTÁGIOS ausentes:', int(semaforos['estagios_num'].isna().sum()))
print('STATUS:', dict(semaforos['STATUS'].value_counts()))

Limpeza dos semáforos concluída
ESTÁGIOS ausentes: 143
STATUS: {'CENTRALIZADO': np.int64(598), 'CONVENCIONAL': np.int64(491), 'DESATIVADO': np.int64(111), 'DETRAN': np.int64(25), 'PROJETO': np.int64(6)}


### 2.3 Reprojeção para coordenadas métricas

Para medir distâncias em metros, reprojetei as duas camadas para EPSG:31984 (UTM 24S).

In [13]:
sinistros = sinistros.to_crs(CRS_METRICO)
semaforos = semaforos.to_crs(CRS_METRICO)
print('CRS atual (deve ser 31984):', sinistros.crs.to_epsg())

CRS atual (deve ser 31984): 31984


### 2.4 Reconstrução do parque semafórico por ano

O dataset de semáforos reflete o parque **atual**, mas os sinistros vão de 2015 a 2024. Para cada
ano, considerei apenas os semáforos que provavelmente já existiam, usando as datas de
implantação/desativação/reativação e o `STATUS`.

**Pressupostos:** implantação ausente = já existia; desativação ausente = nunca desativado;
`STATUS = "PROJETO"` = nunca existiu (excluído); `"DESATIVADO"` sem data = removido (inativo).

In [14]:
def semaforos_ativos_no_ano(semaforos, ano):
    # Subconjunto de semáforos provavelmente ativos no ano informado
    s = semaforos[semaforos['STATUS'] != 'PROJETO'].copy()
    implantacao = s['DATA_IMPLANTAÇÃO'].dt.year
    desativacao = s['DATA_DESATIVAÇÃO'].dt.year
    reativacao  = s['DATA_REATIVAÇÃO'].dt.year
    ja_existia          = implantacao.isna() | (implantacao <= ano)
    foi_desativado      = desativacao.notna() & (desativacao <= ano)
    foi_reativado       = reativacao.notna()  & (reativacao  <= ano)
    desativado_sem_data = (s['STATUS'] == 'DESATIVADO') & s['DATA_DESATIVAÇÃO'].isna()
    return s[ja_existia & ((~foi_desativado) | foi_reativado) & (~desativado_sem_data)]

print('Semáforos ativos em 2015:', len(semaforos_ativos_no_ano(semaforos, 2015)))
print('Semáforos ativos em 2024:', len(semaforos_ativos_no_ano(semaforos, 2024)))

Semáforos ativos em 2015: 754
Semáforos ativos em 2024: 1106


### 2.5 Integração espacial: distância, densidade e interseção

Para cada sinistro, no ano correspondente, calculei:
- `dist_semaforo_m` — distância ao semáforo ativo mais próximo (metros);
- `densidade_semaforos_100m` / `_250m` — nº de semáforos ativos ao redor;
- atributos do semáforo mais próximo (`estágios`, `modo`, `status`);
- `tem_semaforo_interseccao` — 1 se o semáforo mais próximo está a ≤ 30 m.

In [15]:
COLS_SEMAFORO = ['geometry', 'CÓDIGO', 'STATUS', 'MODO_CONTROLE', 'estagios_num']
RENOMEIA = {'CÓDIGO': 'semaforo_proximo_codigo', 'STATUS': 'semaforo_proximo_status',
            'MODO_CONTROLE': 'semaforo_proximo_modo', 'estagios_num': 'semaforo_proximo_estagios'}

def integrar_ano(grupo, ativos):
    grupo = grupo.copy()
    # densidade: nº de semáforos ativos dentro de cada raio (índice espacial, sem scipy)
    indice = ativos.sindex
    pontos = grupo.geometry.values
    for raio in RAIOS_DENSIDADE:
        pares = indice.query(pontos, predicate='dwithin', distance=raio)
        grupo[f'densidade_semaforos_{raio}m'] = np.bincount(pares[0], minlength=len(grupo)).astype('int32')
    # semáforo mais próximo + atributos
    selecao = ativos[COLS_SEMAFORO].rename(columns=RENOMEIA)
    juncao = gpd.sjoin_nearest(grupo, selecao, how='left', distance_col='dist_semaforo_m')
    juncao = juncao[~juncao.index.duplicated(keep='first')]   # desempata vizinhos múltiplos
    return juncao.drop(columns='index_right', errors='ignore')

partes = []
for ano, grupo in sinistros.groupby('ANO', sort=True):
    ativos = semaforos_ativos_no_ano(semaforos, ano)
    partes.append(integrar_ano(grupo, ativos))
sinistros = pd.concat(partes).sort_index()

sinistros['tem_semaforo_interseccao'] = (sinistros['dist_semaforo_m'] <= RAIO_INTERSECCAO_M).astype('int8')
print('Integração concluída |', f'{len(sinistros):,}', 'sinistros enriquecidos')
print(sinistros['dist_semaforo_m'].describe().round(1))

Integração concluída | 130,823 sinistros enriquecidos
count    130823.0
mean        268.9
std         402.7
min           0.0
25%          56.4
50%         147.0
75%         330.6
max        6675.3
Name: dist_semaforo_m, dtype: float64


### 2.6 Dataset final limpo

Removi a geometria (mantendo `LATITUDE`/`LONGITUDE`) e salvei o resultado integrado e limpo em CSV.

In [17]:
dataset_final = pd.DataFrame(sinistros.drop(columns='geometry'))
dataset_final.to_csv('data/sinistros_semaforos_integrado.csv', index=False)
print('Dataset final salvo em: data/sinistros_semaforos_integrado.csv')
print('Dimensões (linhas x colunas):', dataset_final.shape)
dataset_final.sample(5)

Dataset final salvo em: data/sinistros_semaforos_integrado.csv
Dimensões (linhas x colunas): (130823, 35)


,COD_ACIDENTE,DIA,MES,ANO,LOG1,LOG2,LATITUDE,LONGITUDE,TIPOCRUZAMENTO,CONTROLETRAFEGO,...,HORA,data_hora,densidade_semaforos_100m,densidade_semaforos_250m,semaforo_proximo_codigo,semaforo_proximo_status,semaforo_proximo_modo,semaforo_proximo_estagios,dist_semaforo_m,tem_semaforo_interseccao
38517,305009,2,3,2017,"RUA BAR CANINDE,","RUA ELCIAS LOPES,",-3.770473,-38.557426,Cruz,Pare,...,03:15:00,2017-03-02 03:15:00,1,2,509,CONVENCIONAL,LOCAL,3,95.784032,0
113311,TBY2485,30,7,2023,RUA TERTULIANO POTIGUAR,RUA CEL JUCA,-3.745324,-38.493761,Cruz,Pare,...,12:15:00,2023-07-30 12:15:00,0,4,649,CENTRALIZADO,SCOOT CONJUGADO,2,115.294652,0
3090,249457,3,3,2015,"RUA DOS POTIGUARAS, 110",",",-3.719774,-38.516706,Meio de Quadra,Não Há Controle,...,00:00:00,2015-03-03 00:00:00,0,1,286,CENTRALIZADO,ECOTRAFIX CONJUGADO,3,104.363629,0
72597,349569,17,9,2019,"RUA CARLOS CHAGAS, 343",",",-3.780670,-38.583350,Meio de Quadra,Não informado,...,01:19:00,2019-09-17 01:19:00,0,0,527,CONVENCIONAL,LOCAL,3,327.416651,0
81050,361130,6,8,2020,"RUA CUIABA, 1286",",",-3.765081,-38.586254,Meio de Quadra,Não informado,...,03:49:00,2020-08-06 03:49:00,0,1,487,CONVENCIONAL,LOCAL CONJUGADO,2,129.867779,0
